# eRayz Fine-Tuning — sunxds_0.7.8 + Deine BO7/Warzone Bilder
---
Nimmt sunxds_0.7.8 als Basis und trainiert es zusaetzlich auf DEINE Screenshots.
Das Modell kennt dann genau die Skins, Maps und Lichtverhaeltnisse die DU siehst.

**VORBEREITUNG:**
1. Mache 300-500 Screenshots im Spiel (F12 oder Print Screen)
2. Gehe zu https://app.roboflow.com → Kostenloser Account
3. Erstelle ein Projekt: Object Detection, Name: 'bo7-players'
4. Upload deine Screenshots
5. Zeichne Rechtecke um JEDEN Spieler (Klasse: 'player')
   - Koepfe separat labeln als 'head' (optional aber besser)
   - KEIN Loot, KEINE Teammates labeln!
6. Generate Version → Export als 'YOLOv8'
7. Kopiere deinen API Key + Projekt-Infos unten rein

**ANLEITUNG:**
1. Klicke oben auf Laufzeit > Laufzeittyp aendern > GPU (T4)
2. Fuehre ALLE Zellen aus (Shift+Enter)
3. Am Ende wird 'erayz_bo7_v1.onnx' heruntergeladen

In [ ]:
# SCHRITT 1: Installieren
!pip install -q ultralytics roboflow
print('OK')

In [ ]:
# SCHRITT 2: GPU Check
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('KEINE GPU! Gehe zu Laufzeit > Laufzeittyp > GPU')

In [ ]:
# SCHRITT 3: Dein sunxds_0.7.8.pt hochladen
# Klicke links auf das Ordner-Symbol und lade sunxds_0.7.8.pt hoch
# ODER nutze diesen Upload-Dialog:
from google.colab import files
print('Lade jetzt sunxds_0.7.8.pt hoch...')
uploaded = files.upload()
print(f'Hochgeladen: {list(uploaded.keys())}')

In [ ]:
# SCHRITT 4: Roboflow Dataset herunterladen
# ═══════════════════════════════════════════
# HIER DEINE WERTE EINTRAGEN:
# ═══════════════════════════════════════════

RF_API_KEY = 'DEIN_API_KEY'           # Roboflow Settings > API Key
RF_WORKSPACE = 'DEIN_WORKSPACE'       # z.B. 'erayz-gaming'
RF_PROJECT = 'DEIN_PROJEKT'           # z.B. 'bo7-players'
RF_VERSION = 1                         # Version (meistens 1)

# ═══════════════════════════════════════════

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8')
print(f'Dataset: {dataset.location}')

In [ ]:
# SCHRITT 5: Fine-Tuning starten!
# sunxds_0.7.8 als Basis → trainiert auf DEINE Bilder
from ultralytics import YOLO

# Lade das Basis-Modell
model = YOLO('sunxds_0.7.8.pt')
print(f'Basis-Modell geladen: {model.model.yaml}')
print(f'Klassen: {model.names}')

# Fine-Tuning: Weniger Epochen weil wir auf einem guten Modell aufbauen
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=30,              # 30 reichen fuer Fine-Tuning
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=8,
    name='erayz_bo7_v1',
    # Fine-Tuning Settings
    lr0=0.001,              # Niedrigere Lernrate (nicht zu viel aendern)
    lrf=0.01,
    warmup_epochs=2,
    # Augmentation
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.05,
    freeze=10,              # Erste 10 Layer einfrieren (Basis beibehalten)
)
print('Training fertig!')

In [ ]:
# SCHRITT 6: Ergebnisse anzeigen
from IPython.display import Image, display
import os

train_dir = 'runs/detect/erayz_bo7_v1'
if not os.path.exists(train_dir):
    for d in os.listdir('runs/detect'):
        if 'erayz' in d:
            train_dir = f'runs/detect/{d}'
            break

print(f'Ergebnisse: {train_dir}')
if os.path.exists(f'{train_dir}/results.png'):
    display(Image(filename=f'{train_dir}/results.png', width=800))
if os.path.exists(f'{train_dir}/val_batch0_pred.jpg'):
    display(Image(filename=f'{train_dir}/val_batch0_pred.jpg', width=800))

In [ ]:
# SCHRITT 7: ONNX Export
best_model = YOLO(f'{train_dir}/weights/best.pt')

best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
)

import shutil
onnx_src = f'{train_dir}/weights/best.onnx'
final_name = 'erayz_bo7_v1.onnx'
shutil.copy(onnx_src, final_name)

size_mb = os.path.getsize(final_name) / (1024*1024)
print(f'\nFERTIG: {final_name} ({size_mb:.1f} MB)')
print('\nDieses Modell ist sunxds_0.7.8 + DEINE BO7 Bilder!')
print('Lege es in Downloads/backend/ und starte den Aimbot.')

In [ ]:
# SCHRITT 8: Download
from google.colab import files
files.download('erayz_bo7_v1.onnx')
# Auch das .pt speichern (fuer spaeteres Weitertraining)
shutil.copy(f'{train_dir}/weights/best.pt', 'erayz_bo7_v1.pt')
files.download('erayz_bo7_v1.pt')
print('Download gestartet!')